In [278]:
from langchain_chroma import Chroma
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAI, OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
from langchain_core.runnables import (RunnableWithMessageHistory,
                                      RunnableLambda,
                                      RunnablePassthrough)
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.output_parsers import StrOutputParser


from operator import itemgetter


In [279]:
from src.pdf_ingestion import PdfIngestion

In [280]:
ingestor = PdfIngestion(chunk_size=1200,
                        chunk_overlap=180)

pdf_chunks = ingestor.process('C:\Projets_rag_personnels\data\pdf_files\Capstone_FinalReport.pdf')

print(f'Number of chunks : {len(pdf_chunks)}\n')

for c in pdf_chunks:
    c.metadata['source']= 'Capstone_FinalReport.pdf'

for i, chunk in enumerate(pdf_chunks[20:23]):

    print(f'CHUNK {i+1}')
    print(f'source:{chunk.metadata.get('source')}; page:{chunk.metadata.get('page')}')
    print(chunk.page_content,'\n')

<>:4: SyntaxWarning: invalid escape sequence '\P'
<>:4: SyntaxWarning: invalid escape sequence '\P'
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_9408\879778538.py:4: SyntaxWarning: invalid escape sequence '\P'
  pdf_chunks = ingestor.process('C:\Projets_rag_personnels\data\pdf_files\Capstone_FinalReport.pdf')


Number of chunks : 76

CHUNK 1
source:Capstone_FinalReport.pdf; page:8
. A dip in activity may be perceived as an indication of declining customer base. Therefore, temporal analysis is critical in distinguish between these trends. 2.7 Prescriptive Analytics and Decision Support Machine learning and prescriptive analytics have been used in business settings due to their ability to analyse data with intricate relationships (James et al., 2021). In churn prediction, methods like ensemble models have shown promising results (Verbeke et al., 2012). However, it is critical to strike a balance between predictive accuracy and interpretability. Comprehending the drivers of predictions is essential for translating analytical outputs into actionable decisions. 2.8 Research Gap and Contribution While the existing literature has covered the concepts of revenue concentration, churn analysis, demand forecasting and segmentation separately, the reality is that these are interconnected. The contributio

## Embeddings, vector_store and retriever creation

In [281]:
# Loading the openai API Key

import os
from dotenv import load_dotenv

print(os.getenv('OPENAI_API_KEY')[:10]+'...')







sk-proj-rR...


In [282]:
# create an instance of the openai embedding model



embeddings = OpenAIEmbeddings(model= 'text-embedding-3-small')


# Creating a Chroma vector strore

pdf_vector_store_new = Chroma(collection_name='capstone_project_RAG',
                              embedding_function=embeddings,
                              persist_directory='Chroma_capstone_store') 

pdf_vector_store_new.add_documents(pdf_chunks)



print(f'Total number of vector in the store : {pdf_vector_store_new._collection.count()}')


# pdf_vector_store_new.delete_collection()


# Let us build the retriever

pdf_retriever = pdf_vector_store_new.as_retriever(search_kwargs = {'k':4})

Total number of vector in the store : 228


## Create the promp with history

In [ ]:
prompt_conv = ChatPromptTemplate.from_messages([

("system",
"You are an expert assistant who analyzes the RBS Capstone Project Report."
"Your name is Clara."
"Answer questions based on the provided context in the language used by the human"
"If the human greats you, You can also great him and tell him what is your name"
"But you have to avoid any other conversation with him different from the provided concept"),

    MessagesPlaceholder('history'),

    ("user",
    "These are the significant excerpts from the report: \n\n{context}\n\n"
    "My Question:{question}")
 
])

## Building the RAG_CHAIN LCEL

In [294]:
# Lets us build a function to change our context format (from Langchain doc to text)
# secondly we use the RunnableLambda to make the function usable in a Rag Chain pipeline.

def return_only_text(documents):
    return '\n\n...\n\n'.join(doc.page_content for doc in documents)

format_runnable = RunnableLambda(return_only_text)

In [295]:
llm = ChatOpenAI(model = 'gpt-4o-mini')

parser = StrOutputParser()



rag_chain = (
    
    {

    'question':itemgetter('question'),
    'context':itemgetter('question') | pdf_retriever | format_runnable,
    'history':itemgetter('history'),
    }
|prompt_conv
|llm
|parser

)

## Build the Conversational Memory

In [296]:
store_conv = {}


def get_session_history(session_id:str):
    if session_id not in store_conv:
        store_conv[session_id] = ChatMessageHistory()
    return store_conv[session_id]


rag_memory = RunnableWithMessageHistory(

    rag_chain,
    get_session_history,
    input_messages_key='question',
    history_messages_key='history'

)

        

c:\Projets_rag_personnels\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


## Test the RAG conversational Memory

In [299]:
session_id = 'capstone final report'

def ask(question):

    response = rag_memory.invoke(  

        {'question':question},

        config = {'configurable':{'session_id':session_id}}
        
        )
    
    print(f'\n * {question} \n')
   
    print(response,'\n') 
    
    print('='*100)


In [300]:
ask("What is the analysis of revenue concentration states ?")
ask("what are those strategic opportunities and risks ?")
ask("resume this in one sentence.")


 * What is the analysis of revenue concentration states ? 

The analysis of revenue concentration states that the company heavily relies on a small number of high-value clients, specifically about 1,000 "whale" clients, with the top 5% contributing to approximately 74% of total revenue. This extreme concentration indicates a significant risk as the company's revenue is vulnerable to potential changes or losses from these key accounts. Additionally, high-revenue regions, such as the Northwest, serve as critical revenue anchors but also pose risks, while lower-performing regions may offer avenues for growth. The presence of a substantial "Unknown" geographic category further complicates the analysis and highlights the importance of addressing data quality to improve revenue insights. Overall, understanding these dynamics is crucial for interpreting customer churn and revenue risk effectively. 


 * what are those strategic opportunities and risks ? 

The strategic opportunities and risk